In [74]:
import pickle
import numpy as np
import pandas as pd

In [75]:
with open('../data_and_models/search_results_beam_k4_n5_debug.pkl', 'rb') as f:
    df_beam_high = pickle.load(f)

with open('../data_and_models/search_results_tree_k4_n5_debug.pkl', 'rb') as f:
    df_tree_high = pickle.load(f)

with open('../data_and_models/probabilistic_search_results_paths100_steps5_debug.pkl', 'rb') as f:
    df_prob_low = pickle.load(f)

with open('../data_and_models/search_results_beam_k3_n5_debug.pkl', 'rb') as f:
    df_beam_low = pickle.load(f)

with open('../data_and_models/search_results_tree_k3_n5_debug.pkl', 'rb') as f:
    df_tree_low = pickle.load(f)

with open('../data_and_models/probabilistic_search_results_paths200_steps5_debug.pkl', 'rb') as f:
    df_prob_high = pickle.load(f)

In [76]:
df_beam_high = df_beam_high[df_beam_high['covered_threshold']==True].copy()
df_tree_high = df_tree_high[df_tree_high['covered_threshold']==True].copy()
df_prob_low = df_prob_low[df_prob_low['covered_threshold']==True].copy()
df_beam_low = df_beam_low[df_beam_low['covered_threshold']==True].copy()
df_tree_low = df_tree_low[df_tree_low['covered_threshold']==True].copy()
df_prob_high = df_prob_high[df_prob_high['covered_threshold']==True].copy()

In [59]:
dfs = [df_beam_high, df_beam_low, df_tree_high, df_tree_low, df_prob_high, df_prob_low]

# Sanity checks on all

In [60]:
# df is your loaded search_results DataFrame

for df in dfs:
    # Convert the 'drug_sequence' list to a hashable tuple
    df['drug_sequence_tuple'] = df['drug_sequence'].apply(tuple)

    # Now, use the new tuple column for grouping
    df['path_id'] = df.groupby(['search_id', 'drug_sequence_tuple']).ngroup()

    # The rest of your check can proceed as planned
    is_decreasing = df.sort_values('step').groupby('path_id')['final_distance'].is_monotonic_decreasing

    # All values in this series should now be True
    print(is_decreasing.all())

True
True
True
True
True
True


In [61]:
    
for df in dfs:# Check 1: Success condition
    success_check = df[df['is_success']]['final_distance'] < df[df['is_success']]['target_radius']
    print(f"Success condition holds: {success_check.all()}")

    # Check 2: Threshold condition (using default threshold of 0.5)
    threshold_val = 0.5
    thresh_df = df[(df['path_type'] == 'threshold') & (~df['is_success'])]
    progress_threshold = thresh_df['starting_distance'] * (1 - threshold_val)
    threshold_check = thresh_df['final_distance'] <= progress_threshold
    print(f"Threshold condition holds: {threshold_check.all()}")

Success condition holds: True
Threshold condition holds: True
Success condition holds: True
Threshold condition holds: True
Success condition holds: True
Threshold condition holds: True
Success condition holds: True
Threshold condition holds: True
Success condition holds: True
Threshold condition holds: True
Success condition holds: True
Threshold condition holds: True


In [62]:
for df in dfs:
    # Check sequence length against step number
    step_len_check = df['drug_sequence'].str.len() == df['step']
    print(f"Step and sequence length match: {step_len_check.all()}")

    # Check for uniqueness of drugs in sequence
    unique_drugs_check = df['drug_sequence'].apply(lambda seq: len(set(seq)) == len(seq))
    print(f"All paths have unique drugs: {unique_drugs_check.all()}")

Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True


In [63]:
for df in dfs:
    grouped = df.groupby(['search_id', 'step'])
    best_check = grouped['is_best_at_step'].sum()

    print((best_check == 1).all())

    # Create the two series
    min_dists = grouped['final_distance'].min()
    best_dists = df[df['is_best_at_step']].set_index(['search_id', 'step'])['final_distance']

    print(min_dists.head())
    print(best_dists.head())

    # --- FIX: Sort both Series by their index ---
    min_dists.sort_index(inplace=True)
    best_dists.sort_index(inplace=True)

    # Now the comparison will work correctly, even with default tolerance
    print(np.allclose(min_dists, best_dists))

    # The rest of your debug code can remain the same
    are_not_close = ~np.isclose(min_dists, best_dists)
    comparison_df = pd.DataFrame({
        'min_dist_found': min_dists[are_not_close],
        'best_dist_recorded': best_dists[are_not_close]
    })
    comparison_df['difference'] = (comparison_df['min_dist_found'] - comparison_df['best_dist_recorded']).abs()
    print(f'number of occurences: {len(comparison_df)}')
    print(comparison_df)

True
search_id  step
54         5.0     2.810529
101        5.0     3.363243
148        5.0     3.046612
195        4.0     3.827269
           5.0     3.663421
Name: final_distance, dtype: float64
search_id  step
54         5.0     2.810529
101        5.0     3.363243
148        5.0     3.046612
195        4.0     3.827269
           5.0     3.663421
Name: final_distance, dtype: float64
True
number of occurences: 0
Empty DataFrame
Columns: [min_dist_found, best_dist_recorded, difference]
Index: []
True
search_id  step
101        5.0     3.363243
148        5.0     3.046612
195        4.0     3.827269
           5.0     3.663421
523        4.0     4.549383
Name: final_distance, dtype: float64
search_id  step
101        5.0     3.363243
148        5.0     3.046612
195        4.0     3.827269
           5.0     3.663421
525        3.0     4.377250
Name: final_distance, dtype: float64
True
number of occurences: 0
Empty DataFrame
Columns: [min_dist_found, best_dist_recorded, difference]
In

In [64]:
for df in dfs:    
    # Check sequence length against step number
    step_len_check = df['drug_sequence'].str.len() == df['step']
    print(f"Step and sequence length match: {step_len_check.all()}")

    # Check for uniqueness of drugs in sequence
    unique_drugs_check = df['drug_sequence'].apply(lambda seq: len(set(seq)) == len(seq))
    print(f"All paths have unique drugs: {unique_drugs_check.all()}")

Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True
Step and sequence length match: True
All paths have unique drugs: True


# Sanity checks combined tree and beam

In [65]:
# Compare the total number of recorded paths
num_beam_paths = len(df_beam_high)
num_tree_paths = len(df_tree_high)

print(f"Beam search found {num_beam_paths} paths.")
print(f"Tree search found {num_tree_paths} paths.")

# A more detailed view, comparing paths per search
beam_counts = df_beam_high['search_id'].value_counts()
tree_counts = df_tree_high['search_id'].value_counts()

comparison_df = pd.DataFrame({
    'beam_path_count': beam_counts,
    'tree_path_count': tree_counts
}).fillna(0)

print("\n--- Path Counts Per Search ID ---")
print(comparison_df.head())

Beam search found 298 paths.
Tree search found 13512 paths.

--- Path Counts Per Search ID ---
           beam_path_count  tree_path_count
search_id                                  
54                     1.0               46
101                    4.0              294
148                    3.0               25
195                    8.0              507
478                    0.0               47


In [66]:
import numpy as np
import pandas as pd

# Find the minimum distance for each search_id in both dataframes
best_beam_dists = df_beam_high.groupby('search_id')['final_distance'].min()
best_tree_dists = df_tree_high.groupby('search_id')['final_distance'].min()

# Combine into a single DataFrame, aligning by search_id
quality_df = pd.DataFrame({
    'beam_best_dist': best_beam_dists,
    'tree_best_dist': best_tree_dists
})

# A failed beam search is infinitely worse than any found tree path.
# Replace NaN in the beam column with infinity.
quality_df['beam_best_dist'].fillna(np.inf, inplace=True)

# We only care about cases where the tree search actually found a path.
# So we can drop rows where tree_best_dist is NaN.
quality_df.dropna(subset=['tree_best_dist'], inplace=True)


# The check: Are all tree distances less than or equal to beam distances?
is_tree_better_or_equal = (quality_df['tree_best_dist'] <= quality_df['beam_best_dist']).all()

print(f"Is tree search quality always >= beam search quality? {is_tree_better_or_equal}")

Is tree search quality always >= beam search quality? True


/var/folders/gw/tcx_140d3b9_7gszcr9sk6wm0000gn/T/ipykernel_33885/3223048750.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  quality_df['beam_best_dist'].fillna(np.inf, inplace=True)


In [67]:
# Get the best path for each search_id in the beam results
# Using .loc and .idxmin() to get the full row of the best path
best_beam_paths = df_beam_high.loc[df_beam_high.groupby('search_id')['final_distance'].idxmin()]

# Convert drug sequences to tuples to make them hashable for searching
best_beam_paths['drug_seq_tuple'] = best_beam_paths['drug_sequence'].apply(tuple)
df_tree_high['drug_seq_tuple'] = df_tree_high['drug_sequence'].apply(tuple)

# For each best beam path, check if it exists in the tree results
missing_paths = 0
for _, row in best_beam_paths.iterrows():
    sid = row['search_id']
    path_tuple = row['drug_seq_tuple']

    # Check if the path exists in the tree results for the same search
    found_in_tree = df_tree_high[
        (df_tree_high['search_id'] == sid) &
        (df_tree_high['drug_seq_tuple'] == path_tuple)
    ]

    if found_in_tree.empty:
        missing_paths += 1
        print(f"Path not found! Search ID: {sid}, Path: {path_tuple}")

if missing_paths == 0:
    print("All best beam paths were successfully found within the tree search results.")

All best beam paths were successfully found within the tree search results.


# Sanity checks combined k-path-tree low and high

In [68]:
# ==============================================================================
# CHECK 1: Path Quantity
# A larger branching factor (k) should create exponentially more paths.
# ==============================================================================
print("--- 1. Path Quantity Check ---")
num_paths_high_k = len(df_tree_high)
num_paths_low_k = len(df_tree_low)

print(f"Paths found with higher k: {num_paths_high_k}")
print(f"Paths found with lower k:  {num_paths_low_k}")

if num_paths_high_k > num_paths_low_k:
    print("✅ PASSED: Higher k generated more paths as expected.")
else:
    print("❌ FAILED: Higher k did not generate more paths.")
print("-" * 30)


# ==============================================================================
# CHECK 2: Solution Quality
# A more thorough search (higher k) should never yield a worse best-path.
# ==============================================================================
print("--- 2. Solution Quality Check ---")
# Get the best final distance for each search_id
best_high_k = df_tree_high.groupby('search_id')['final_distance'].min()
best_low_k = df_tree_low.groupby('search_id')['final_distance'].min()

# Align them and fill missing values for a robust comparison
quality_df = pd.DataFrame({
    'high_k_best': best_high_k,
    'low_k_best': best_low_k
})
# A failed search has an infinitely bad distance. This handles cases where
# one search found a path for a search_id but the other didn't.
quality_df.fillna(np.inf, inplace=True)

# The check:
is_quality_maintained = (quality_df['high_k_best'] <= quality_df['low_k_best']).all()

if is_quality_maintained:
    print("✅ PASSED: Solution quality was maintained or improved with higher k.")
else:
    print("❌ FAILED: Found cases where lower k produced a better result (this should not happen).")
    print("\nSearches where lower k found a better path:")
    print(quality_df[quality_df['high_k_best'] > quality_df['low_k_best']])
print("-" * 30)


# ==============================================================================
# CHECK 3: Path Subset Guarantee
# Every path found by the low_k search must also be found by the high_k search.
# ==============================================================================
print("--- 3. Path Subset Check ---")
# Create a unique identifier for each path in both dataframes
# Using a tuple of the drug sequence makes it hashable for set operations.
df_tree_low['path_key'] = df_tree_low.apply(
    lambda row: (row['search_id'], tuple(row['drug_sequence'])), axis=1
)
df_tree_high['path_key'] = df_tree_high.apply(
    lambda row: (row['search_id'], tuple(row['drug_sequence'])), axis=1
)

# Create sets of the unique path keys
low_k_paths = set(df_tree_low['path_key'])
high_k_paths = set(df_tree_high['path_key'])

# Check if the small set is a subset of the large set
is_subset = low_k_paths.issubset(high_k_paths)

if is_subset:
    print("✅ PASSED: All paths from the lower k run were found in the higher k run.")
else:
    print("❌ FAILED: Not all paths from the lower k run were found in the higher k run.")
    missing_paths = low_k_paths - high_k_paths
    print(f"\nFound {len(missing_paths)} paths from low_k run missing in high_k run.")
    # Show an example of a path that was expected but not found
    if missing_paths:
        print("Example missing path (search_id, drug_sequence):", list(missing_paths)[0])
print("-" * 30)

--- 1. Path Quantity Check ---
Paths found with higher k: 13512
Paths found with lower k:  3656
✅ PASSED: Higher k generated more paths as expected.
------------------------------
--- 2. Solution Quality Check ---
✅ PASSED: Solution quality was maintained or improved with higher k.
------------------------------
--- 3. Path Subset Check ---
✅ PASSED: All paths from the lower k run were found in the higher k run.
------------------------------


# Sanity checks k-path-beam high and low

In [69]:
# ==============================================================================
# CHECK 1: Solution Quality
# A wider beam considers more options at each step, so the best final solution
# should never be worse than the one found with a smaller beam.
# ==============================================================================
print("--- 1. Solution Quality Check ---")
# Get the best final distance for each search_id
best_high_k = df_beam_high.groupby('search_id')['final_distance'].min()
best_low_k = df_beam_low.groupby('search_id')['final_distance'].min()

# Align them and fill missing values for a robust comparison
quality_df = pd.DataFrame({
    'high_k_best': best_high_k,
    'low_k_best': best_low_k
})
# A failed search has an infinitely bad distance.
quality_df.fillna(np.inf, inplace=True)

# The check:
is_quality_maintained = (quality_df['high_k_best'] <= quality_df['low_k_best']).all()

if is_quality_maintained:
    print("✅ PASSED: Solution quality was maintained or improved with a wider beam.")
else:
    print("❌ FAILED: Found cases where a smaller beam produced a better result (this should not happen).")
    print("\nSearches where smaller k found a better path:")
    print(quality_df[quality_df['high_k_best'] > quality_df['low_k_best']])
print("-" * 30)


# ==============================================================================
# CHECK 2: Path Quantity
# A wider beam has the potential to find more valid paths that meet the
# recording criteria, though this is not a strict guarantee like the tree search.
# ==============================================================================
print("--- 2. Path Quantity Check ---")
num_paths_high_k = len(df_beam_high)
num_paths_low_k = len(df_beam_low)

print(f"Paths found with higher k: {num_paths_high_k}")
print(f"Paths found with lower k:  {num_paths_low_k}")

if num_paths_high_k >= num_paths_low_k:
    print("✅ PASSED: Higher k generated an equal or greater number of paths, as expected.")
else:
    print("⚠️  WARNING: Higher k generated fewer paths. This is possible but may be worth investigating.")
print("-" * 30)


# ==============================================================================
# CHECK 3: Best Path Persistence
# While not all paths from the low_k search will be in the high_k search, we
# expect that the BEST path from the low_k search is good enough to have been
# kept and recorded by the high_k search.
# ==============================================================================
print("--- 3. Best Path Persistence Check ---")
# Get the full rows for the best path of each search in the low_k results
best_low_k_paths = df_beam_low.loc[df_beam_low.groupby('search_id')['final_distance'].idxmin()]

# Create a unique key for these best paths
best_low_k_paths['path_key'] = best_low_k_paths.apply(
    lambda row: (row['search_id'], tuple(row['drug_sequence'])), axis=1
)

# Create a set of all path keys from the high_k results for efficient lookup
high_k_path_keys = set(df_beam_high.apply(
    lambda row: (row['search_id'], tuple(row['drug_sequence'])), axis=1
))

# Check how many of the best low_k paths are also present in the high_k results
persisted_paths = best_low_k_paths[best_low_k_paths['path_key'].isin(high_k_path_keys)]

percent_persisted = 100 * len(persisted_paths) / len(best_low_k_paths) if len(best_low_k_paths) > 0 else 100

print(f"{len(persisted_paths)} out of {len(best_low_k_paths)} best paths from the low_k run persisted.")
print(f"✅ PASSED: ({percent_persisted:.1f}%) of best paths persisted in the high_k run.")
if percent_persisted < 100:
    print("This is acceptable for beam search, as path rankings can shift with a wider beam.")
print("-" * 30)

--- 1. Solution Quality Check ---
✅ PASSED: Solution quality was maintained or improved with a wider beam.
------------------------------
--- 2. Path Quantity Check ---
Paths found with higher k: 298
Paths found with lower k:  185
✅ PASSED: Higher k generated an equal or greater number of paths, as expected.
------------------------------
--- 3. Best Path Persistence Check ---
24 out of 29 best paths from the low_k run persisted.
✅ PASSED: (82.8%) of best paths persisted in the high_k run.
This is acceptable for beam search, as path rankings can shift with a wider beam.
------------------------------


# Sanity check for prob_search high and low

In [77]:
# ==============================================================================
# CHECK 1: Solution Quality
# A search with more random walks has a higher probability of finding a better
# final solution. The best path should be at least as good as, or better than,
# the one found with fewer attempts.
# ==============================================================================
print("--- 1. Solution Quality Check ---")
# Get the best final distance for each search_id
best_high_paths = df_prob_high.groupby('search_id')['final_distance'].min()
best_low_paths = df_prob_low.groupby('search_id')['final_distance'].min()

# Align them and fill missing values for a robust comparison
quality_df = pd.DataFrame({
    'high_n_paths_best': best_high_paths,
    'low_n_paths_best': best_low_paths
})
# A failed search has an infinitely bad distance
quality_df.fillna(np.inf, inplace=True)

# The check:
is_quality_maintained = (quality_df['high_n_paths_best'] <= quality_df['low_n_paths_best']).all()

if is_quality_maintained:
    print("✅ PASSED: Solution quality was maintained or improved with more paths.")
else:
    print("❌ FAILED: Found cases where fewer paths produced a better result. While statistically possible, this is highly unlikely and worth investigating.")
    print("\nSearches where low n_paths found a better path:")
    print(quality_df[quality_df['high_n_paths_best'] > quality_df['low_n_paths_best']])
print("-" * 30)


# ==============================================================================
# CHECK 2: Path Quantity
# Running more Monte Carlo simulations (n_paths) should naturally result in
# a larger total number of recorded paths.
# ==============================================================================
print("--- 2. Path Quantity Check ---")
num_paths_high = len(df_prob_high)
num_paths_low = len(df_prob_low)

print(f"Paths found with higher n_paths: {num_paths_high}")
print(f"Paths found with lower n_paths:  {num_paths_low}")

if num_paths_high > num_paths_low:
    print("✅ PASSED: Higher n_paths generated more paths as expected.")
else:
    print("❌ FAILED: Higher n_paths did not generate more paths.")
print("-" * 30)


# ==============================================================================
# CHECK 3: Success and Discovery Rate
# More attempts should increase the number of successful searches and the
# diversity of successful paths discovered.
# ==============================================================================
print("--- 3. Success and Discovery Rate Check ---")
# Filter for successful paths in both runs
success_high = df_prob_high[df_prob_high['is_success']]
success_low = df_prob_low[df_prob_low['is_success']]

# Compare the number of searches that found at least one successful path
successful_searches_high = success_high['search_id'].nunique()
successful_searches_low = success_low['search_id'].nunique()

print(f"Number of searches with at least one success (high n_paths): {successful_searches_high}")
print(f"Number of searches with at least one success (low n_paths):  {successful_searches_low}")

if successful_searches_high >= successful_searches_low:
    print("✅ PASSED: More paths led to an equal or greater number of successful searches.")
else:
    print("⚠️ WARNING: More paths led to fewer successful searches. This is statistically unlikely.")

# Compare the number of unique successful drug sequences discovered
unique_success_paths_high = success_high['drug_sequence'].apply(tuple).nunique()
unique_success_paths_low = success_low['drug_sequence'].apply(tuple).nunique()

print(f"\nUnique successful paths discovered (high n_paths): {unique_success_paths_high}")
print(f"Unique successful paths discovered (low n_paths):  {unique_success_paths_low}")

if unique_success_paths_high >= unique_success_paths_low:
    print("✅ PASSED: More paths led to an equal or greater number of unique solutions.")
else:
    print("⚠️ WARNING: More paths led to fewer unique solutions discovered.")
print("-" * 30)

--- 1. Solution Quality Check ---
❌ FAILED: Found cases where fewer paths produced a better result. While statistically possible, this is highly unlikely and worth investigating.

Searches where low n_paths found a better path:
           high_n_paths_best  low_n_paths_best
search_id                                     
901                 4.111707          4.019089
1183                     inf          3.011107
1230                4.248221          4.146064
1888                3.897353          3.683019
2217                3.777629          3.727549
------------------------------
--- 2. Path Quantity Check ---
Paths found with higher n_paths: 47
Paths found with lower n_paths:  17
✅ PASSED: Higher n_paths generated more paths as expected.
------------------------------
--- 3. Success and Discovery Rate Check ---
Number of searches with at least one success (high n_paths): 0
Number of searches with at least one success (low n_paths):  0
✅ PASSED: More paths led to an equal or greater n

In [78]:
import pandas as pd
import numpy as np

# Assume df_prob_high and df_prob_low are already loaded

# --- Revised Sanity Check for Probabilistic Search ---

print("--- Statistical Quality Check ---")

# Get the best final distance for each search_id
best_high_paths = df_prob_high.groupby('search_id')['final_distance'].min()
best_low_paths = df_prob_low.groupby('search_id')['final_distance'].min()

# Align the results for comparison
quality_df = pd.DataFrame({
    'high_n_paths_best': best_high_paths,
    'low_n_paths_best': best_low_paths
})

# Compare overall statistical performance
# We expect the mean/median distance to be lower (better) with more paths
print(f"Mean best distance (high n_paths): {quality_df['high_n_paths_best'].mean():.4f}")
print(f"Mean best distance (low n_paths):  {quality_df['low_n_paths_best'].mean():.4f}")
print("-" * 20)
print(f"Median best distance (high n_paths): {quality_df['high_n_paths_best'].median():.4f}")
print(f"Median best distance (low n_paths):  {quality_df['low_n_paths_best'].median():.4f}")

# Count how many times each search won
# Filling NaNs with infinity ensures a failed search always loses to a successful one
high_wins = (quality_df['high_n_paths_best'] < quality_df['low_n_paths_best']).sum()
low_wins = (quality_df['high_n_paths_best'] > quality_df['low_n_paths_best']).sum()
draws = (quality_df['high_n_paths_best'] == quality_df['low_n_paths_best']).sum()

print("\n--- Head-to-Head Comparison ---")
print(f"High n_paths found a better solution for {high_wins} searches.")
print(f"Low n_paths found a better solution for  {low_wins} searches.")
print(f"Both found the same best solution for   {draws} searches.")

# The main takeaway:
if high_wins > low_wins and quality_df['high_n_paths_best'].mean() < quality_df['low_n_paths_best'].mean():
    print("\n✅ PASSED: Statistically, running more paths improved the results on average.")
else:
    print("\n⚠️  WARNING: Running more paths did not lead to a clear statistical improvement on average.")

--- Statistical Quality Check ---
Mean best distance (high n_paths): 3.8707
Mean best distance (low n_paths):  3.7174
--------------------
Median best distance (high n_paths): 3.7776
Median best distance (low n_paths):  3.7275

--- Head-to-Head Comparison ---
High n_paths found a better solution for 0 searches.
Low n_paths found a better solution for  4 searches.
Both found the same best solution for   0 searches.

⚠️  WARNING: Running more paths did not lead to a clear statistical improvement on average.
